# W01c - Dataset with User-Defined Extended Features

In [1]:
### Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from dataset_operations import get_title_code, get_position_code, get_edu_degree_major_code, get_company_code, get_main_skills_code
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',None)
# print(dir(pd.options.display))

In [2]:
### Load the salary dataset with the correct date, plus the country coefficient dataset
salaries = pd.read_csv('Salaries_v4_202412161100.csv')
countries_coef = pd.read_csv('countries_salary_coef.csv')
### Drop the duplicate rows with equal id's and those with company score and education score being 0.0
salaries = salaries.drop_duplicates('id').reset_index()
salaries = salaries.drop(salaries[salaries['company_score'] == 0.0].index, axis=0)
salaries = salaries.drop(salaries[salaries['education_score'] == 0.0].index, axis=0)
salaries = salaries.drop('index',axis=1)

In [3]:
print("THIS SALARY DATASET CONTAINS {} ROWS & {} COLUMNS".format(salaries.shape[0], salaries.shape[1]))

THIS SALARY DATASET CONTAINS 19554 ROWS & 16 COLUMNS


In [4]:
salaries.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 19554 entries, 0 to 19556
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 19554 non-null  object 
 1   found_country      19554 non-null  object 
 2   title              19542 non-null  object 
 3   position           19554 non-null  object 
 4   employment_type    12397 non-null  object 
 5   company            16936 non-null  object 
 6   company_score      15739 non-null  float64
 7   edu_degrees        19000 non-null  object 
 8   edu_degrees_major  18623 non-null  object 
 9   working_year       19554 non-null  int64  
 10  education_score    14956 non-null  float64
 11  ms_counts          19554 non-null  int64  
 12  skill_counts       19554 non-null  int64  
 13  main_skills        19554 non-null  object 
 14  skills             19554 non-null  object 
 15  amount_usd         19554 non-null  int64  
dtypes: float64(2), int64(4

In [5]:
salaries.isnull().sum()

id                      0
found_country           0
title                  12
position                0
employment_type      7157
company              2618
company_score        3815
edu_degrees           554
edu_degrees_major     931
working_year            0
education_score      4598
ms_counts               0
skill_counts            0
main_skills             0
skills                  0
amount_usd              0
dtype: int64

In [6]:
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000


In [7]:
### Make a new dataframe to hold the salary coefficients of each feature and each code
features_codes_coefs_df = pd.DataFrame(columns=['feature','code','coef'])

## CREATE NEW FEATURES

### 1. FOUND COUNTRY

In [8]:
### Create a new feature to store the country coefficients of all rows
country_coef = [];
for country in salaries['found_country']:
    if type(country) == float:   
        country_coef.append(0.0);   continue
    else:
        country_coef.append(countries_coef[countries_coef['en_name'] == country]['salary_coefficient'].values[0])
salaries['country_coef'] = country_coef

In [9]:
print("#### COUNTS OF ALL FOUND COUNTRIES ####")
print(salaries['found_country'].value_counts())

#### COUNTS OF ALL FOUND COUNTRIES ####
United States     17852
United Kingdom     1702
Name: found_country, dtype: int64


### 2. TITLE

In [10]:
### Create a title code list to represent all the titles as in short codes
title_code = []
### Count each identified title of all rows, and collect their corresponding salaries
### Null titles and those indicating openness to work and opportunities
title_null = 0;           salary_null = [];
title_open_to_work = 0;   salary_open_to_work = []; 
### White collar titles
white_collar_titles_cnt = [0 for _ in range(79)]
white_collar_titles_slr = [[] for _ in range(79)]
# print(white_collar_titles_cnt)
# print(white_collar_titles_slr)
### Blue collar titles
blue_collar_titles_cnt = [0 for _ in range(57)]
blue_collar_titles_slr = [[] for _ in range(57)]
# print(blue_collar_titles_cnt)
# print(blue_collar_titles_slr)
### Uncategorized number of titles
title_uncategorized = 0;   salary_uncategorized = [];   title_uncategorized_examples = [];

In [11]:
all_titles = salaries['title'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_titles)):
    title = all_titles[i];   salary = all_salaries[i]
    code, index = get_title_code(title)
    title_code.append(code)
    if code.startswith('WC'):
        white_collar_titles_cnt[index] += 1;   white_collar_titles_slr[index].append(salary);
    elif code.startswith('BC'):
        blue_collar_titles_cnt[index] += 1;   blue_collar_titles_slr[index].append(salary);
    elif code == 'NO_TITLE':
        title_null += 1;   salary_null.append(salary);
    elif code == 'OPEN':
        title_open_to_work += 1;   salary_open_to_work.append(salary);
    elif code == 'UNCT':
        title_uncategorized += 1;   salary_uncategorized.append(salary);   title_uncategorized_examples.append(title);
salaries['title_code'] = title_code

In [12]:
print("#### COUNTS OF ALL WHITE COLLAR TITLES ####")
print(white_collar_titles_cnt, '-->', sum(white_collar_titles_cnt))
print("#### COUNTS OF ALL BLUE COLLAR TITLES ####")
print(blue_collar_titles_cnt, '-->', sum(blue_collar_titles_cnt))
print("#### {} OPEN TO WORK TITLES ####".format(title_open_to_work))
print("#### {} NULL TITLES ####".format(title_null))
print("#### {} UNCATEGORIZED TITLES ####".format(title_uncategorized))

#### COUNTS OF ALL WHITE COLLAR TITLES ####
[15, 22, 477, 187, 24, 26, 549, 172, 48, 218, 33, 23, 169, 35, 10, 102, 25, 969, 20, 154, 1602, 85, 57, 88, 284, 18, 2, 20, 198, 19, 64, 167, 11, 150, 40, 76, 180, 16, 79, 19, 874, 48, 74, 12, 19, 379, 7, 28, 3, 29, 8, 19, 352, 29, 497, 296, 6, 6, 310, 5, 83, 138, 12, 5, 195, 21, 4, 7, 3787, 42, 5, 81, 130, 85, 120, 17, 273, 508, 32] --> 14999
#### COUNTS OF ALL BLUE COLLAR TITLES ####
[44, 184, 5, 60, 141, 56, 52, 8, 23, 17, 15, 31, 46, 48, 100, 26, 8, 19, 38, 83, 11, 17, 16, 92, 95, 9, 55, 102, 78, 185, 5, 106, 46, 19, 50, 32, 77, 78, 108, 115, 8, 99, 77, 44, 88, 95, 75, 4, 97, 207, 50, 4, 72, 103, 15, 41, 14] --> 3393
#### 40 OPEN TO WORK TITLES ####
#### 12 NULL TITLES ####
#### 1110 UNCATEGORIZED TITLES ####


In [13]:
### Get the salary averages of all titles
white_collar_titles_slr_avg = [];   blue_collar_titles_slr_avg = [];
print("#### ALL WHITE COLLAR TITLES AVG SALARY ####")
for i in range(len(white_collar_titles_slr)):
    white_collar_titles_slr_avg.append(round(np.array(white_collar_titles_slr[i]).mean(),2))
print(white_collar_titles_slr_avg)
print("#### ALL BLUE COLLAR TITLES AVG SALARY ####")
for i in range(len(blue_collar_titles_slr)):
    blue_collar_titles_slr_avg.append(round(np.array(blue_collar_titles_slr[i]).mean(),2))
print(blue_collar_titles_slr_avg)

salary_open_to_work_avg = round(np.array(salary_open_to_work).mean(),2)
print("#### OPEN TO WORK TITLES AVG SALARY = {} ####".format(salary_open_to_work_avg))
salary_null_avg = round(np.array(salary_null).mean(),2)
print("#### NULL TITLES AVG SALARY = {} ####".format(salary_null_avg))
salary_uncategorized_avg = round(np.array(salary_uncategorized).mean(),2)
print("#### UNCATEGORIZED TITLES AVG SALARY  = {} ####".format(salary_uncategorized_avg))

#### ALL WHITE COLLAR TITLES AVG SALARY ####
[219000.0, 172212.09, 174621.73, 251460.57, 287563.21, 173653.85, 126472.79, 181554.76, 115500.0, 114601.15, 132536.61, 140365.87, 137120.05, 109685.71, 102400.0, 166588.24, 251360.0, 129742.21, 200399.15, 200284.06, 201224.12, 153435.29, 220263.16, 206340.6, 180292.32, 305020.22, 129500.0, 112450.0, 147484.06, 104210.53, 101125.0, 158293.41, 196090.91, 212472.33, 125883.65, 108026.32, 344594.44, 170440.0, 129420.72, 136646.89, 217833.13, 116666.67, 194567.57, 162416.67, 89157.89, 180368.63, 127928.57, 93625.32, 86333.33, 191896.55, 89193.38, 156736.84, 248690.73, 145551.72, 231557.34, 161622.27, 149308.0, 112652.83, 150065.62, 139800.0, 79481.93, 162253.62, 141833.33, 152128.6, 268621.54, 61666.67, 91496.5, 123000.0, 176260.89, 138798.21, 263000.0, 108666.67, 148295.81, 143541.18, 169687.68, 148705.88, 143843.67, 174728.35, 106317.44]
#### ALL BLUE COLLAR TITLES AVG SALARY ####
[76636.36, 142396.74, 113600.0, 106400.0, 217312.06, 76589.29, 

In [14]:
### Determine the maximum average salary among all titles
all_titles_slr_avg = np.concatenate((np.array(white_collar_titles_slr_avg), np.array(blue_collar_titles_slr_avg),
        np.array(salary_open_to_work_avg), np.array(salary_null_avg), np.array(salary_uncategorized_avg)), axis=None)
all_titles_slr_max_avg = all_titles_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_titles_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 360948.05 ####


In [15]:
### Make a dictionary to store all salary coefficients of all titles and assign those coefficients as a new feature
title_salary_avg_dict = {}
for i in range(len(white_collar_titles_slr_avg)):
    title_salary_avg_dict["WC"+str(i).zfill(2)] = round(white_collar_titles_slr_avg[i] / all_titles_slr_max_avg, 3)
for i in range(len(blue_collar_titles_slr_avg)):
    title_salary_avg_dict["BC"+str(i).zfill(2)] = round(blue_collar_titles_slr_avg[i] / all_titles_slr_max_avg, 3)
# title_salary_avg_dict['OPEN'] = round(salary_open_to_work_avg / all_titles_slr_max_avg, 3)
title_salary_avg_dict['OPEN'] = 0.200
# title_salary_avg_dict['NO_TITLE'] = round(salary_null_avg / all_titles_slr_max_avg, 3)
title_salary_avg_dict['NO_TITLE'] = 0.200
title_salary_avg_dict['UNCT'] = round(salary_uncategorized_avg / all_titles_slr_max_avg, 3)
# print(title_salary_avg_dict)

### Add these title codes and salary coefficients to the dataframe
for code, coef in title_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['title'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

title_slr_coef = []
for code in salaries['title_code']:
    title_slr_coef.append(title_salary_avg_dict[code])
salaries['title_slr_coef'] = title_slr_coef

### 3. TITLE SENIORITY

In [16]:
### Create a list to store the title seniority types of all rows
title_seniority = []
### Get counts and salaries of each predefined title seniority
tt_senior_cnt = 0;         tt_senior_slr = [];         tt_staff_cnt = 0;         tt_staff_slr = [];  
tt_senior_staff_cnt = 0;   tt_senior_staff_slr = [];   tt_principal_cnt = 0;     tt_principal_slr = [];        
tt_associate_cnt = 0;      tt_associate_slr = [];      tt_junior_cnt = 0;        tt_junior_slr = [];
tt_no_seniority_cnt = 0;   tt_no_seniority_slr = [];   tt_null_cnt = 0;          tt_null_slr = [];
all_titles = salaries['title'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_titles)):
    title = all_titles[i];   salary = all_salaries[i]
    if type(title) == float:   
        title_seniority.append('NO_TITLE');   tt_null_cnt += 1;   tt_null_slr.append(salary);   continue
    elif 'principal' in title.lower():    
        title_seniority.append('PRINCIPAL');   tt_principal_cnt += 1;   tt_principal_slr.append(salary);
    elif 'senior staff' in title.lower() or 'sr. staff' in title.lower():   
        title_seniority.append('SENIOR_STAFF');   tt_senior_staff_cnt += 1;   tt_senior_staff_slr.append(salary);
    elif 'staff' in title.lower():        
        title_seniority.append('STAFF');   tt_staff_cnt += 1;    tt_staff_slr.append(salary);
    elif 'senior' in title.lower() or 'sr.' in title.lower():   
        title_seniority.append('SENIOR');   tt_senior_cnt += 1;   tt_senior_slr.append(salary);
    elif 'associate' in title.lower():    
        title_seniority.append('ASSOCIATE');   tt_associate_cnt += 1;   tt_associate_slr.append(salary);
    elif 'junior' in title.lower():       
        title_seniority.append('JUNIOR');   tt_junior_cnt += 1;      tt_junior_slr.append(salary);
    else:   
        title_seniority.append('NO_SENIORITY');   tt_no_seniority_cnt += 1;    tt_no_seniority_slr.append(salary);
salaries['title_seniority'] = title_seniority

In [17]:
print("#### COUNTS OF ALL TITLE SENIORITIES ####")
print(salaries['title_seniority'].value_counts())

#### COUNTS OF ALL TITLE SENIORITIES ####
NO_SENIORITY    16607
SENIOR           2048
PRINCIPAL         350
STAFF             263
ASSOCIATE         237
SENIOR_STAFF       21
JUNIOR             16
NO_TITLE           12
Name: title_seniority, dtype: int64


In [18]:
### Get the salary averages of all title seniorities
print("#### ALL TITLE SENIORITIES AVG SALARY ####")
tt_senior_slr_avg = round(np.array(tt_senior_slr).mean(),2)
tt_staff_slr_avg = round(np.array(tt_staff_slr).mean(),2)
tt_senior_staff_slr_avg = round(np.array(tt_senior_staff_slr).mean(),2)
tt_principal_slr_avg = round(np.array(tt_principal_slr).mean(),2)
tt_associate_slr_avg = round(np.array(tt_associate_slr).mean(),2)
tt_junior_slr_avg = round(np.array(tt_junior_slr).mean(),2)
tt_no_seniority_slr_avg = round(np.array(tt_no_seniority_slr).mean(),2)
tt_null_slr_avg = round(np.array(tt_null_slr).mean(),2)
print("SENIOR       = {}".format(tt_senior_slr_avg))
print("STAFF        = {}".format(tt_staff_slr_avg))
print("SENIOR STAFF = {}".format(tt_senior_staff_slr_avg))
print("PRINCIPAL    = {}".format(tt_principal_slr_avg))
print("ASSOCIATE    = {}".format(tt_associate_slr_avg))
print("JUNIOR       = {}".format(tt_junior_slr_avg))
print("NO SENIORITY = {}".format(tt_no_seniority_slr_avg))
print("NO TITLE     = {}".format(tt_null_slr_avg))

#### ALL TITLE SENIORITIES AVG SALARY ####
SENIOR       = 204567.22
STAFF        = 246153.8
SENIOR STAFF = 319400.24
PRINCIPAL    = 259810.77
ASSOCIATE    = 101836.58
JUNIOR       = 65705.38
NO SENIORITY = 155747.25
NO TITLE     = 121268.08


In [19]:
### Get the maximum average salary among all title seniorities
all_title_seniority_slr_avg = np.array([tt_senior_slr_avg, tt_staff_slr_avg, tt_senior_staff_slr_avg, tt_principal_slr_avg,
        tt_associate_slr_avg, tt_junior_slr_avg, tt_no_seniority_slr_avg, tt_null_slr_avg])
# print(all_title_seniority_slr_avg)
all_title_seniority_slr_max_avg = all_title_seniority_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_title_seniority_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 319400.24 ####


In [20]:
### Make a dictionary to store all salary coefficients of all title seniorities
### Assign those coefficients as a new feature
title_snr_salary_avg_dict = {}
title_snr_salary_avg_dict['SENIOR'] = round(tt_senior_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['STAFF'] = round(tt_staff_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['SENIOR_STAFF'] = round(tt_senior_staff_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['PRINCIPAL'] = round(tt_principal_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['ASSOCIATE'] = round(tt_associate_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['JUNIOR'] = round(tt_junior_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['NO_SENIORITY'] = round(tt_no_seniority_slr_avg / all_title_seniority_slr_max_avg, 3)
# title_snr_salary_avg_dict['NO_TITLE'] = round(tt_null_slr_avg / all_title_seniority_slr_max_avg, 3)
title_snr_salary_avg_dict['NO_TITLE'] = 0.200
# print(title_snr_salary_avg_dict)

### Add these title seniority codes and salary coefficients to the dataframe
for code, coef in title_snr_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['title_snr'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

title_seniority_slr_coef = []
for seniority in salaries['title_seniority']:
    title_seniority_slr_coef.append(title_snr_salary_avg_dict[seniority])
salaries['title_seniority_slr_coef'] = title_seniority_slr_coef

### 4. POSITION

In [21]:
### Create a position code list to represent all the positions as in short codes
position_code = []
### Count each identified position of all rows, and collect their corresponding salaries
### Thankfully, there is neither null nor 'open-to-work' positions
### White collar positions in the list
white_collar_positions_cnt = [0 for _ in range(77)]
white_collar_positions_slr = [[] for _ in range(77)]
# print(white_collar_positions_cnt)
# print(white_collar_positions_slr)
### Blue collar positions in the list
blue_collar_positions_cnt = [0 for _ in range(52)]
blue_collar_positions_slr = [[] for _ in range(52)]
# print(blue_collar_positions_cnt)
# print(blue_collar_positions_slr)
# Uncategorized number of positions
position_uncategorized = 0;   position_uncategorized_salary = [];   position_uncategorized_examples = [];

In [22]:
all_positions = salaries['position'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_positions)):
    pos = all_positions[i];   salary = all_salaries[i];
    code, index = get_position_code(pos)
    position_code.append(code)
    if code.startswith('WC'):
        white_collar_positions_cnt[index] += 1;   white_collar_positions_slr[index].append(salary);
    elif code.startswith('BC'):
        blue_collar_positions_cnt[index] += 1;   blue_collar_positions_slr[index].append(salary);
    elif code == 'UNCT':
        position_uncategorized += 1;  position_uncategorized_salary.append(salary);   position_uncategorized_examples.append(pos);
salaries['position_code'] = position_code

In [23]:
print("#### COUNTS OF ALL WHITE COLLAR POSITIONS ####")
print(white_collar_positions_cnt, '-->', sum(white_collar_positions_cnt))
print("#### COUNTS OF ALL BLUE COLLAR POSITIONS ####")
print(blue_collar_positions_cnt, '-->', sum(blue_collar_positions_cnt))
print("#### {} UNCATEGORIZED POSITIONS ####".format(position_uncategorized))

#### COUNTS OF ALL WHITE COLLAR POSITIONS ####
[20, 18, 160, 6, 18, 392, 167, 41, 215, 18, 187, 23, 26, 106, 29, 1067, 21, 156, 1775, 82, 59, 114, 279, 19, 12, 195, 22, 64, 172, 138, 30, 80, 174, 33, 3, 16, 27, 50, 12, 949, 34, 66, 12, 27, 398, 26, 12, 32, 25, 32, 347, 501, 303, 4, 12, 110, 10, 85, 137, 10, 201, 16, 2, 7, 4, 169, 4041, 274, 56, 8, 88, 109, 69, 122, 29, 461, 29] --> 14843
#### COUNTS OF ALL BLUE COLLAR POSITIONS ####
[202, 60, 134, 65, 46, 47, 30, 43, 12, 18, 46, 58, 100, 34, 26, 48, 84, 18, 98, 96, 47, 54, 97, 95, 190, 6, 110, 44, 22, 50, 94, 81, 122, 9, 6, 95, 81, 112, 99, 101, 78, 7, 5, 96, 207, 48, 5, 77, 107, 13, 61, 18] --> 3502
#### 1209 UNCATEGORIZED POSITIONS ####


In [24]:
### Get the salary averages of all positions
white_collar_positions_slr_avg = [];   blue_collar_positions_slr_avg = [];
print("#### ALL WHITE COLLAR POSITIONS AVG SALARY ####")
for i in range(len(white_collar_positions_slr)):
    white_collar_positions_slr_avg.append(round(np.array(white_collar_positions_slr[i]).mean(),2))
print(white_collar_positions_slr_avg)
print("#### ALL BLUE COLLAR POSITIONS AVG SALARY ####")
for i in range(len(blue_collar_positions_slr)):
    blue_collar_positions_slr_avg.append(round(np.array(blue_collar_positions_slr[i]).mean(),2))
print(blue_collar_positions_slr_avg)

salary_uncategorized_avg = round(np.array(position_uncategorized_salary).mean(),2)
print("#### UNCATEGORIZED POSITIONS AVG SALARY  = {} ####".format(salary_uncategorized_avg))

#### ALL WHITE COLLAR POSITIONS AVG SALARY ####
[189300.0, 163745.61, 191744.37, 106726.33, 171555.56, 121055.9, 184679.16, 111951.22, 115093.02, 127430.61, 138935.54, 133478.26, 100269.23, 167547.17, 262827.59, 128691.49, 211054.81, 202045.95, 207121.88, 152829.27, 221355.93, 224210.53, 180631.11, 304966.53, 85750.0, 148579.49, 87409.09, 100812.5, 160738.37, 215135.95, 120790.87, 108025.0, 343008.72, 78000.0, 207661.0, 170440.0, 177333.33, 103228.66, 134666.67, 216130.8, 84882.35, 193136.36, 159736.58, 88148.15, 177825.31, 194461.54, 102168.58, 91281.25, 158280.0, 174312.5, 248948.52, 231497.01, 163439.58, 180750.0, 134416.67, 138995.35, 210262.1, 78905.88, 161291.97, 137600.0, 264608.31, 62625.0, 99500.0, 127285.71, 90938.5, 161471.56, 175901.14, 142979.66, 139663.79, 219000.0, 107920.45, 151399.33, 137586.45, 169552.07, 145207.52, 174587.85, 115655.17]
#### ALL BLUE COLLAR POSITIONS AVG SALARY ####
[144039.6, 106550.0, 217395.52, 77430.77, 96521.74, 41212.77, 93566.67, 45116.28, 802

In [25]:
### Determine the maximum average salary among all positions
all_positions_slr_avg = np.concatenate((np.array(white_collar_positions_slr_avg), np.array(blue_collar_positions_slr_avg),
        np.array(salary_uncategorized_avg)), axis=None)
all_positions_slr_max_avg = all_positions_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_positions_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 357648.44 ####


In [26]:
### Make a dictionary to store all salary coefficients of all positions and assign those coefficients as a new feature
position_salary_avg_dict = {}
for i in range(len(white_collar_positions_slr_avg)):
    position_salary_avg_dict["WC"+str(i).zfill(2)] = round(white_collar_positions_slr_avg[i] / all_positions_slr_max_avg, 3)
for i in range(len(blue_collar_positions_slr_avg)):
    position_salary_avg_dict["BC"+str(i).zfill(2)] = round(blue_collar_positions_slr_avg[i] / all_positions_slr_max_avg, 3)
position_salary_avg_dict['UNCT'] = round(salary_uncategorized_avg / all_positions_slr_max_avg, 3)
# print(position_salary_avg_dict)

### Add these position codes and salary coefficients to the dataframe
for code, coef in position_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['position'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

position_slr_coef = []
for code in salaries['position_code']:
    position_slr_coef.append(position_salary_avg_dict[code])
salaries['position_slr_coef'] = position_slr_coef

### 5. POSITION SENIORITY

In [27]:
### Create a list to store the position seniority types of all rows
position_seniority = []
### Get the counts of each unique seniority in all positions and collect their salaries
pos_principal_cnt = 0;         pos_principal_slr = [];         pos_senior_staff_cnt = 0;     pos_senior_staff_slr = [];  
pos_staff_cnt = 0;             pos_staff_slr = [];             pos_senior_cnt = 0;           pos_senior_slr = [];  
pos_associate_cnt = 0;         pos_associate_slr = [];         pos_junior_cnt = 0;           pos_junior_slr = [];  
pos_no_seniority_cnt = 0;      pos_no_seniority_slr = [];
all_positions = salaries['position'].values
all_salaries = salaries['amount_usd'].values
position_seniority_coef = []
for i in range(len(all_positions)):
    pos = all_positions[i];   salary = all_salaries[i];
    if 'principal' in pos.lower():     
        position_seniority.append('PRINCIPAL');   pos_principal_cnt += 1;    pos_principal_slr.append(salary);
    elif 'senior staff' in pos.lower() or 'sr. staff' in pos.lower():   
        position_seniority.append('SENIOR_STAFF');   pos_senior_staff_cnt += 1;   pos_senior_staff_slr.append(salary);
    elif 'staff' in pos.lower():       
        position_seniority.append('STAFF');   pos_staff_cnt += 1;    pos_staff_slr.append(salary);
    elif 'senior' in pos.lower() or 'sr.' in pos.lower():     
        position_seniority.append('SENIOR');   pos_senior_cnt += 1;    pos_senior_slr.append(salary);
    elif 'associate' in pos.lower():   
        position_seniority.append('ASSOCIATE');   pos_associate_cnt += 1;    pos_associate_slr.append(salary);
    elif 'junior' in pos.lower():      
        position_seniority.append('JUNIOR');   pos_junior_cnt += 1;    pos_junior_slr.append(salary);
    else:
        position_seniority.append('NO_SENIORITY');   pos_no_seniority_cnt += 1;    pos_no_seniority_slr.append(salary);
salaries['position_seniority'] = position_seniority

In [28]:
print("#### COUNTS OF ALL POSITION SENIORITIES ####")
print(salaries['position_seniority'].value_counts())

#### COUNTS OF ALL POSITION SENIORITIES ####
NO_SENIORITY    15888
SENIOR           2562
PRINCIPAL         413
STAFF             372
ASSOCIATE         271
SENIOR_STAFF       29
JUNIOR             19
Name: position_seniority, dtype: int64


In [29]:
### Get the salary averages of all position seniorities
print("#### ALL POSITION SENIORITIES AVG SALARY ####")
pos_senior_slr_avg = round(np.array(pos_senior_slr).mean(),2)
pos_staff_slr_avg = round(np.array(pos_staff_slr).mean(),2)
pos_senior_staff_slr_avg = round(np.array(pos_senior_staff_slr).mean(),2)
pos_principal_slr_avg = round(np.array(pos_principal_slr).mean(),2)
pos_associate_slr_avg = round(np.array(pos_associate_slr).mean(),2)
pos_junior_slr_avg = round(np.array(pos_junior_slr).mean(),2)
pos_no_seniority_slr_avg = round(np.array(pos_no_seniority_slr).mean(),2)
print("SENIOR       = {}".format(pos_senior_slr_avg))
print("STAFF        = {}".format(pos_staff_slr_avg))
print("SENIOR STAFF = {}".format(pos_senior_staff_slr_avg))
print("PRINCIPAL    = {}".format(pos_principal_slr_avg))
print("ASSOCIATE    = {}".format(pos_associate_slr_avg))
print("JUNIOR       = {}".format(pos_junior_slr_avg))
print("NO SENIORITY = {}".format(pos_no_seniority_slr_avg))

#### ALL POSITION SENIORITIES AVG SALARY ####
SENIOR       = 202856.24
STAFF        = 246261.33
SENIOR STAFF = 296508.62
PRINCIPAL    = 253892.91
ASSOCIATE    = 100880.98
JUNIOR       = 73513.32
NO SENIORITY = 153634.86


In [30]:
### Get the maximum average salary among all position seniorities
all_position_seniority_slr_avg = np.array([pos_senior_slr_avg, pos_staff_slr_avg, pos_senior_staff_slr_avg, pos_principal_slr_avg,
        pos_associate_slr_avg, pos_junior_slr_avg, pos_no_seniority_slr_avg])
# print(all_position_seniority_slr_avg)
all_position_seniority_slr_max_avg = all_position_seniority_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_position_seniority_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 296508.62 ####


In [31]:
### Make a dictionary to store all salary coefficients of all position seniorities
### Assign those coefficients as a new feature
pos_snr_salary_avg_dict = {}
pos_snr_salary_avg_dict['SENIOR'] = round(pos_senior_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['STAFF'] = round(pos_staff_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['SENIOR_STAFF'] = round(pos_senior_staff_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['PRINCIPAL'] = round(pos_principal_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['ASSOCIATE'] = round(pos_associate_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['JUNIOR'] = round(pos_junior_slr_avg / all_position_seniority_slr_max_avg, 3)
pos_snr_salary_avg_dict['NO_SENIORITY'] = round(pos_no_seniority_slr_avg / all_position_seniority_slr_max_avg, 3)
# print(pos_snr_salary_avg_dict)

### Add these position seniority codes and salary coefficients to the dataframe
for code, coef in pos_snr_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['position_snr'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

position_seniority_slr_coef = []
for seniority in salaries['position_seniority']:
    position_seniority_slr_coef.append(pos_snr_salary_avg_dict[seniority])
salaries['position_seniority_slr_coef'] = position_seniority_slr_coef

### 6. EMPLOYMENT TYPE

In [32]:
### Assign the fixed coefficients of employment types as a different feature
employment_type_coef = []
for emp in salaries['employment_type']:
    if type(emp) == float:         employment_type_coef.append(0.10);   continue
    elif emp == 'Full-time' or emp == 'Permanent Full-time' or emp == 'Permanent':    employment_type_coef.append(0.99)
    elif emp == 'Part-time':       employment_type_coef.append(0.60)
    elif emp == 'Contract':        employment_type_coef.append(0.50)
    elif emp == 'Self-employed':   employment_type_coef.append(0.45)
    elif emp == 'Freelance':       employment_type_coef.append(0.45)
    elif emp == 'Apprenticeship':  employment_type_coef.append(0.35)
    elif emp == 'Internship':      employment_type_coef.append(0.30)
    elif emp == 'Seasonal':        employment_type_coef.append(0.40)
    else:   employment_type_coef.append(0.10)
salaries['employment_type_coef'] = employment_type_coef

### 7. EMPLOYMENT TYPE (SALARY)

In [33]:
### Create a list to store the employment types of all rows in cleaner format
emp_type = []
### Get the counts of each unique employment types and collect their salaries
emp_full_time_cnt = 0;      emp_full_time_slr = [];      emp_part_time_cnt = 0;         emp_part_time_slr = [];
emp_contract_cnt = 0;       emp_contract_slr = [];       emp_self_employed_cnt = 0;     emp_self_employed_slr = [];
emp_freelance_cnt = 0;      emp_freelance_slr = [];      emp_apprenticeship_cnt = 0;    emp_apprenticeship_slr = [];
emp_internship_cnt = 0;     emp_internship_slr = [];     emp_seasonal_cnt = 0;          emp_seasonal_slr = [];
emp_type_null_cnt = 0;      emp_type_null_slr = [];      emp_type_unknown_cnt = 0;      emp_type_unknown_slr = [];
all_emp_types = salaries['employment_type'].values
all_salaries = salaries['amount_usd'].values
employment_type_slr_coef = []
for i in range(len(all_emp_types)):
    emp = all_emp_types[i];   salary = all_salaries[i];
    if type(emp) == float:
        emp_type.append('NO_EMP_TYPE');   emp_type_null_cnt += 1;   emp_type_null_slr.append(salary);   continue
    elif emp == 'Full-time' or emp == 'Permanent Full-time' or emp == 'Permanent':
        emp_type.append('FULL_TIME');   emp_full_time_cnt += 1;   emp_full_time_slr.append(salary);
    elif emp == 'Part-time':
        emp_type.append('PART_TIME');   emp_part_time_cnt += 1;   emp_part_time_slr.append(salary);
    elif emp == 'Contract':
        emp_type.append('CONTRACT');   emp_contract_cnt += 1;    emp_contract_slr.append(salary);
    elif emp == 'Self-employed':
        emp_type.append('SELF_EMPLOYED');   emp_self_employed_cnt += 1;   emp_self_employed_slr.append(salary);
    elif emp == 'Freelance':
        emp_type.append('FREELANCE');   emp_freelance_cnt += 1;   emp_freelance_slr.append(salary);
    elif emp == 'Apprenticeship':
        emp_type.append('APPRENTICE');   emp_apprenticeship_cnt += 1;   emp_apprenticeship_slr.append(salary);
    elif emp == 'Internship':
        emp_type.append('INTERN');   emp_internship_cnt += 1;   emp_internship_slr.append(salary);
    elif emp == 'Seasonal':
        emp_type.append('SEASONAL');   emp_seasonal_cnt += 1;   emp_seasonal_slr.append(salary);
    else:
        emp_type.append('UNKNOWN');   emp_type_unknown_cnt += 1;   emp_type_unknown_slr.append(salary);
salaries['emp_type'] = emp_type

In [34]:
print("#### COUNTS OF ALL EMPLOYMENT TYPES ####")
print(salaries['emp_type'].value_counts())

#### COUNTS OF ALL EMPLOYMENT TYPES ####
FULL_TIME        11917
NO_EMP_TYPE       7157
CONTRACT           290
SELF_EMPLOYED       67
PART_TIME           54
FREELANCE           29
APPRENTICE          20
INTERN              12
SEASONAL             6
UNKNOWN              2
Name: emp_type, dtype: int64


In [35]:
### Get the salary averages of all employment types
print("#### ALL EMPLOYMENT TYPES AVERAGE SALARY ####")
emp_full_time_slr_avg = round(np.array(emp_full_time_slr).mean(),2)
emp_part_time_slr_avg = round(np.array(emp_part_time_slr).mean(),2)
emp_contract_slr_avg = round(np.array(emp_contract_slr).mean(),2)
emp_self_employed_slr_avg = round(np.array(emp_self_employed_slr).mean(),2)
emp_freelance_slr_avg = round(np.array(emp_freelance_slr).mean(),2)
emp_apprenticeship_slr_avg = round(np.array(emp_apprenticeship_slr).mean(),2)
emp_internship_slr_avg = round(np.array(emp_internship_slr).mean(),2)
emp_seasonal_slr_avg = round(np.array(emp_seasonal_slr).mean(),2)
emp_type_null_slr_avg = round(np.array(emp_type_null_slr).mean(),2)
emp_type_unknown_slr_avg = round(np.array(emp_type_unknown_slr).mean(),2)
print("FULL_TIME     = {}".format(emp_full_time_slr_avg))
print("PART_TIME     = {}".format(emp_part_time_slr_avg))
print("CONTRACT      = {}".format(emp_contract_slr_avg))
print("SELF_EMPLOYED = {}".format(emp_self_employed_slr_avg))
print("FREELANCE     = {}".format(emp_freelance_slr_avg))
print("APPRENTICE    = {}".format(emp_apprenticeship_slr_avg))
print("INTERN        = {}".format(emp_internship_slr_avg))
print("SEASONAL      = {}".format(emp_seasonal_slr_avg))
print("NO_EMP_TYPE   = {}".format(emp_type_null_slr_avg))
print("UNKNOWN       = {}".format(emp_type_unknown_slr_avg))

#### ALL EMPLOYMENT TYPES AVERAGE SALARY ####
FULL_TIME     = 160837.36
PART_TIME     = 108514.61
CONTRACT      = 130039.53
SELF_EMPLOYED = 121905.19
FREELANCE     = 120784.62
APPRENTICE    = 103314.55
INTERN        = 111321.42
SEASONAL      = 130333.33
NO_EMP_TYPE   = 170158.31
UNKNOWN       = 261000.0


In [36]:
### Get the maximum average salary among all employment types (except unknowns and nulls)
all_emp_type_slr_avg = np.array([emp_full_time_slr_avg, emp_part_time_slr_avg, emp_contract_slr_avg, emp_self_employed_slr_avg,
        emp_freelance_slr_avg, emp_apprenticeship_slr_avg, emp_internship_slr_avg, emp_seasonal_slr_avg])
# print(all_emp_type_slr_avg)
all_emp_type_slr_max_avg = all_emp_type_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_emp_type_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 160837.36 ####


In [37]:
### Make a dictionary to store all salary coefficients of all employment types
### Assign those coefficients as a new feature
emp_type_salary_avg_dict = {}
emp_type_salary_avg_dict['FULL_TIME'] = round(emp_full_time_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['PART_TIME'] = round(emp_part_time_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['CONTRACT'] = round(emp_contract_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['SELF_EMPLOYED'] = round(emp_self_employed_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['FREELANCE'] = round(emp_freelance_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['APPRENTICE'] = round(emp_apprenticeship_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['INTERN'] = round(emp_internship_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['SEASONAL'] = round(emp_seasonal_slr_avg / all_emp_type_slr_max_avg, 3)
# emp_type_salary_avg_dict['NO_EMP_TYPE'] = round(emp_type_null_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['NO_EMP_TYPE'] = 0.200
# emp_type_salary_avg_dict['UNKNOWN'] = round(emp_type_unknown_slr_avg / all_emp_type_slr_max_avg, 3)
emp_type_salary_avg_dict['UNKNOWN'] = 0.200
# print(emp_type_salary_avg_dict)

### Add these employment type codes and salary coefficients to the dataframe
for code, coef in emp_type_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['emp_type'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

employment_type_slr_coef = []
for emp_type in salaries['emp_type']:
    employment_type_slr_coef.append(emp_type_salary_avg_dict[emp_type])
salaries['employment_type_slr_coef'] = employment_type_slr_coef

### 8. HIGHEST EDUCATION DEGREE

In [38]:
### Create two lists to store the highest education degree name for all rows and give the fixed coefficients
highest_edu_deg_name = [];   highest_edu_deg_lvl = []
for edu_deg in salaries['edu_degrees']:
    if type(edu_deg) == float:
        highest_edu_deg_name.append('NO_DEGREE');   highest_edu_deg_lvl.append(0.100);   continue
    if 'PHD' in edu_deg:
        highest_edu_deg_name.append('PHD');   highest_edu_deg_lvl.append(1.000);
    elif 'MASTERS' in edu_deg:
        highest_edu_deg_name.append('MASTERS');   highest_edu_deg_lvl.append(0.750);
    elif 'UNDERGRADUATE' in edu_deg:
        highest_edu_deg_name.append('UNDERGRADUATE');   highest_edu_deg_lvl.append(0.500);
    elif 'ASSOCIATE' in edu_deg:
        highest_edu_deg_name.append('ASSOCIATE');   highest_edu_deg_lvl.append(0.350);
    elif 'HIGH_SCHOOL' in edu_deg:
        highest_edu_deg_name.append('HIGH_SCHOOL');   highest_edu_deg_lvl.append(0.200);
salaries['highest_edu_deg_name'] = highest_edu_deg_name
salaries['highest_edu_deg_lvl'] = highest_edu_deg_lvl

In [39]:
print("#### COUNTS OF HIGHEST EDUCATION DEGREES ####")
print(salaries['highest_edu_deg_name'].value_counts())

#### COUNTS OF HIGHEST EDUCATION DEGREES ####
UNDERGRADUATE    10097
MASTERS           7386
PHD               1237
NO_DEGREE          554
ASSOCIATE          216
HIGH_SCHOOL         64
Name: highest_edu_deg_name, dtype: int64


### 9. HIGHEST EDUCATION DEGREE (SALARY)

In [40]:
### Count the education degrees for all rows and collect the corresponding salaries
highest_edu_deg_phd = 0;           edu_phd_slr = [];           highest_edu_deg_master = 0;      edu_master_slr = [];
highest_edu_deg_undergrad = 0;     edu_undergrad_slr = [];     highest_edu_deg_associate = 0;   edu_associate_slr = [];
highest_edu_deg_high_school = 0;   edu_high_school_slr = [];   null_edu_deg = 0;                null_edu_deg_slr = [];
all_highest_edu_deg_names = salaries['highest_edu_deg_name'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_highest_edu_deg_names)):
    deg = all_highest_edu_deg_names[i];   salary = all_salaries[i];
    if deg == 'NO_DEGREE':   
        null_edu_deg += 1;   null_edu_deg_slr.append(salary);   
    elif deg == 'PHD':
        highest_edu_deg_phd += 1;   edu_phd_slr.append(salary);
    elif deg == 'MASTERS':
        highest_edu_deg_master += 1;   edu_master_slr.append(salary);
    elif deg == 'UNDERGRADUATE':
        highest_edu_deg_undergrad += 1;   edu_undergrad_slr.append(salary);
    elif deg == 'ASSOCIATE':
        highest_edu_deg_associate += 1;   edu_associate_slr.append(salary);
    elif deg == 'HIGH_SCHOOL':
        highest_edu_deg_high_school += 1;   edu_high_school_slr.append(salary);

In [41]:
### Get the salary averages of all education degrees
print("#### ALL EDUCATION DEGREES AVERAGE SALARY ####")
edu_phd_slr_avg = round(np.array(edu_phd_slr).mean(),2)
edu_master_slr_avg = round(np.array(edu_master_slr).mean(),2)
edu_undergrad_slr_avg = round(np.array(edu_undergrad_slr).mean(),2)
edu_associate_slr_avg = round(np.array(edu_associate_slr).mean(),2)
edu_high_school_slr_avg = round(np.array(edu_high_school_slr).mean(),2)
edu_null_slr_avg = round(np.array(null_edu_deg_slr).mean(),2)
print("PHD           = {}".format(edu_phd_slr_avg))
print("MASTERS       = {}".format(edu_master_slr_avg))
print("UNDERGRADUATE = {}".format(edu_undergrad_slr_avg))
print("ASSOCIATE     = {}".format(edu_associate_slr_avg))
print("HIGH_SCHOOL   = {}".format(edu_high_school_slr_avg))
print("NO_DEGREE     = {}".format(edu_null_slr_avg))

#### ALL EDUCATION DEGREES AVERAGE SALARY ####
PHD           = 227906.17
MASTERS       = 175446.08
UNDERGRADUATE = 149826.04
ASSOCIATE     = 104027.65
HIGH_SCHOOL   = 95494.28
NO_DEGREE     = 135974.4


In [42]:
### Get the maximum average salary among all education degrees
all_edu_deg_slr_avg = np.array([edu_phd_slr_avg, edu_master_slr_avg, edu_undergrad_slr_avg, edu_associate_slr_avg,
        edu_high_school_slr_avg, edu_null_slr_avg])
# print(all_edu_deg_slr_avg)
all_edu_deg_slr_max_avg = all_edu_deg_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_edu_deg_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 227906.17 ####


In [43]:
### Make a dictionary to store all salary coefficients of all education degrees
### Assign those coefficients as a new feature
edu_deg_salary_avg_dict = {}
edu_deg_salary_avg_dict['PHD'] = round(edu_phd_slr_avg / all_edu_deg_slr_max_avg, 3)
edu_deg_salary_avg_dict['MASTERS'] = round(edu_master_slr_avg / all_edu_deg_slr_max_avg, 3)
edu_deg_salary_avg_dict['UNDERGRADUATE'] = round(edu_undergrad_slr_avg / all_edu_deg_slr_max_avg, 3)
edu_deg_salary_avg_dict['ASSOCIATE'] = round(edu_associate_slr_avg / all_edu_deg_slr_max_avg, 3)
edu_deg_salary_avg_dict['HIGH_SCHOOL'] = round(edu_high_school_slr_avg / all_edu_deg_slr_max_avg, 3)
# edu_deg_salary_avg_dict['NO_DEGREE'] = round(edu_null_slr_avg / all_edu_deg_slr_max_avg, 3)
edu_deg_salary_avg_dict['NO_DEGREE'] = 0.200
# print(edu_deg_salary_avg_dict)

### Add these education degree codes and salary coefficients to the dataframe
for code, coef in edu_deg_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['edu_degree'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

edu_deg_slr_coef = []
for edu_deg in salaries['highest_edu_deg_name']:
    edu_deg_slr_coef.append(edu_deg_salary_avg_dict[edu_deg])
salaries['edu_deg_slr_coef'] = edu_deg_slr_coef

### 10. EDUCATION DEGREE MAJOR

In [44]:
### Create a education degree major code list to represent all the education degree majors as in short codes
edu_degree_major_code = []
### Count each identified education degree major of all rows, and collect their corresponding salaries
edu_degree_null = 0;   edu_degree_salary_null = [];
edu_degree_major_cnt = [0 for _ in range(57)]
edu_degree_major_slr = [[] for _ in range(57)]
# print(edu_degree_major_cnt)
# print(edu_degree_major_slr)
# Uncategorized number of education degree majors
edu_degree_uncategorized = 0;   edu_degree_uncategorized_salary = [];   edu_degree_uncategorized_examples = [];

In [45]:
all_edu_degrees_majors = salaries['edu_degrees_major'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_edu_degrees_majors)):
    edu = all_edu_degrees_majors[i];   salary = all_salaries[i];   edm_code = ""
    edm_code, indices = get_edu_degree_major_code(edu)
    edm_code = edm_code.strip().replace(' ',',')
    edu_degree_major_code.append(edm_code)
    if edm_code.startswith('EDM'):
        for j in range(len(indices)):
            edu_degree_major_cnt[indices[j]] += 1;   edu_degree_major_slr[indices[j]].append(salary);
    elif edm_code == 'NO_DEGREE':
        edu_degree_null += 1;   edu_degree_salary_null.append(salary);
    elif edm_code == 'UNCT':
        edu_degree_uncategorized += 1;   edu_degree_uncategorized_salary.append(salary);   edu_degree_uncategorized_examples.append(edu)
salaries['edu_degree_major_code'] = edu_degree_major_code

In [46]:
print("#### COUNTS OF ALL EDUCATION DEGREE MAJORS ####")
print(edu_degree_major_cnt, '-->', sum(edu_degree_major_cnt))
print("#### {} NULL EDUCATION DEGREE MAJORS ####".format(edu_degree_null))
print("#### {} UNCATEGORIZED EDUCATION DEGREE MAJORS ####".format(edu_degree_uncategorized))

#### COUNTS OF ALL EDUCATION DEGREE MAJORS ####
[401, 38, 70, 91, 1458, 12, 281, 1805, 463, 3, 189, 271, 284, 37, 1572, 107, 71, 4729, 180, 230, 686, 103, 895, 1614, 111, 149, 16, 832, 169, 63, 1868, 15, 206, 108, 58, 218, 186, 739, 1197, 424, 36, 68, 222, 119, 44, 976, 573, 240, 542, 57, 5, 5, 145, 541, 635, 29, 38] --> 26224
#### 931 NULL EDUCATION DEGREE MAJORS ####
#### 3001 UNCATEGORIZED EDUCATION DEGREE MAJORS ####


In [47]:
### Get the salary averages of all education degree majors
edu_degree_major_slr_avg = [];
print("#### ALL EDUCATION DEGREE MAJORS AVG SALARY ####")
for i in range(len(edu_degree_major_slr)):
   edu_degree_major_slr_avg.append(round(np.array(edu_degree_major_slr[i]).mean(),2))
print(edu_degree_major_slr_avg)

edu_degree_null_salary_avg = round(np.array(edu_degree_salary_null).mean(),2)
print("#### NULL EDUCATION DEGREE MAJORS AVG SALARY = {} ####".format(edu_degree_null_salary_avg))
edu_degree_uncategorized_salary_avg = round(np.array(edu_degree_uncategorized_salary).mean(),2)
print("#### UNCATEGORIZED EDUCATION DEGREE MAJORS AVG SALARY  = {} ####".format(edu_degree_uncategorized_salary_avg))

#### ALL EDUCATION DEGREE MAJORS AVG SALARY ####
[140904.49, 161338.32, 171794.76, 166061.36, 152780.35, 192482.42, 145914.88, 154662.12, 165900.41, 202333.33, 168305.99, 165947.59, 135294.48, 107702.7, 192511.32, 174355.71, 167817.65, 186055.5, 164515.65, 153173.02, 176381.97, 168328.51, 175114.98, 198720.45, 179988.34, 126268.25, 77062.5, 168678.5, 158660.29, 170984.13, 171233.91, 162866.67, 200727.85, 121780.46, 170699.05, 146007.42, 197503.49, 141465.09, 187511.93, 192888.26, 186509.69, 72647.06, 126744.79, 233788.97, 176796.73, 223420.27, 188482.09, 172893.61, 129440.5, 117262.54, 189800.0, 94000.0, 134702.97, 177044.19, 199607.49, 99551.72, 155583.74]
#### NULL EDUCATION DEGREE MAJORS AVG SALARY = 140435.79 ####
#### UNCATEGORIZED EDUCATION DEGREE MAJORS AVG SALARY  = 138077.14 ####


In [48]:
### Determine the maximum average salary among all education degree majors
all_edu_degrees_slr_avg = np.concatenate((np.array(edu_degree_major_slr_avg), np.array(edu_degree_null_salary_avg),
        np.array(edu_degree_uncategorized_salary_avg)), axis=None)
all_edu_degrees_slr_max_avg = all_edu_degrees_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_edu_degrees_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 233788.97 ####


In [49]:
### Make a dictionary to store all salary coefficients of all education degree majors
### Assign those coefficients as two new features (average & maximum)
edu_degree_major_salary_avg_dict = {}
for i in range(len(edu_degree_major_slr_avg)):
    edu_degree_major_salary_avg_dict["EDM"+str(i).zfill(2)] = round(edu_degree_major_slr_avg[i] / all_edu_degrees_slr_max_avg, 3)
# edu_degree_major_salary_avg_dict['NO_DEGREE'] = round(edu_degree_null_salary_avg / all_edu_degrees_slr_max_avg ,3)
edu_degree_major_salary_avg_dict['NO_DEGREE'] = 0.200
edu_degree_major_salary_avg_dict['UNCT'] = round(edu_degree_uncategorized_salary_avg / all_edu_degrees_slr_max_avg, 3)
# print(edu_degree_major_salary_avg_dict)

### Add these education degree major codes and salary coefficients to the dataframe
for code, coef in edu_degree_major_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['edu_degree_major'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

edu_degree_major_slr_coef_avg = [];   edu_degree_major_slr_coef_max = [];
for code in salaries['edu_degree_major_code']:
    edm_codes = code.split(',');   coefs = []
    for edm in edm_codes:
        coefs.append(edu_degree_major_salary_avg_dict[edm])
    edu_degree_major_slr_coef_avg.append(round(np.array(coefs).mean(), 3))
    edu_degree_major_slr_coef_max.append(round(np.array(coefs).max(), 3))
salaries['edu_degree_major_slr_coef_avg'] = edu_degree_major_slr_coef_avg
salaries['edu_degree_major_slr_coef_max'] = edu_degree_major_slr_coef_max

### 11. COMPANY

In [50]:
### Create a company code list to represent all identified comapnies as in short codes
company_code = []
### Count each identified comapny of all rows, and collect their corresponding salaries
company_null = 0;   company_null_salary = [];
company_cnt = [0 for _ in range(68)]
company_slr = [[] for _ in range(68)]
# print(company_cnt)
# print(company_slr)
# Uncategorized number of companies
company_uncategorized = 0;   company_uncategorized_salary = [];   company_uncategorized_examples = [];

In [51]:
all_companies = salaries['company'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_companies)):
    company = all_companies[i];   salary = all_salaries[i];
    code, index = get_company_code(company)
    company_code.append(code)
    if code.startswith('CMP'):
        company_cnt[index] += 1;   company_slr[index].append(salary);
    elif code == 'NO_COMPANY':
        company_null += 1;   company_null_salary.append(salary);
    elif code == 'UNCT':
        company_uncategorized += 1;   company_uncategorized_salary.append(salary);
salaries['company_code'] = company_code

In [52]:
print("#### COUNTS OF ALL COMPANIES ####")
print(company_cnt, '-->', sum(company_cnt))
print("#### {} NULL COMPANIES ####".format(company_null))
print("#### {} UNCATEGORIZED COMPANIES ####".format(company_uncategorized))

#### COUNTS OF ALL COMPANIES ####
[1369, 1290, 885, 724, 735, 684, 592, 494, 507, 446, 427, 418, 355, 291, 270, 241, 234, 210, 198, 182, 176, 188, 170, 165, 158, 148, 137, 113, 109, 103, 92, 91, 89, 85, 85, 85, 78, 76, 64, 62, 58, 57, 56, 50, 49, 49, 49, 46, 47, 45, 43, 44, 44, 43, 41, 40, 39, 43, 38, 37, 35, 34, 33, 38, 30, 30, 34, 30] --> 14008
#### 2618 NULL COMPANIES ####
#### 2928 UNCATEGORIZED COMPANIES ####


In [53]:
### Get the salary averages of all identified companies
company_slr_avg = [];
print("#### ALL COMPANIES AVG SALARY ####")
for i in range(len(company_slr)):
   company_slr_avg.append(round(np.array(company_slr[i]).mean(),2))
print(company_slr_avg)

company_null_salary_avg = round(np.array(company_null_salary).mean(),2)
print("#### NULL COMPANIES AVG SALARY = {} ####".format(company_null_salary_avg))
company_uncategorized_salary_avg = round(np.array(company_uncategorized_salary).mean(),2)
print("#### UNCATEGORIZED COMPANIES AVG SALARY  = {} ####".format(company_uncategorized_salary_avg))

#### ALL COMPANIES AVG SALARY ####
[175778.16, 250374.77, 160464.11, 200279.04, 208273.54, 204295.3, 172142.23, 188546.25, 149624.79, 136819.42, 296363.0, 163380.38, 183848.9, 147082.47, 110529.63, 224531.12, 206380.34, 172594.97, 205505.05, 262258.24, 261687.5, 102265.96, 235323.53, 184624.24, 185873.42, 121105.7, 124969.46, 158292.04, 124770.64, 117009.71, 159065.22, 120032.97, 129277.24, 135917.65, 56813.98, 163929.41, 94256.41, 127552.63, 183500.0, 111919.35, 87812.91, 277824.56, 127553.57, 153900.0, 105857.14, 94081.63, 114244.33, 199413.04, 94702.13, 117422.22, 67255.81, 226590.91, 94659.09, 81046.51, 100790.71, 71523.45, 127307.69, 187116.28, 92552.63, 139648.65, 110771.43, 53617.65, 135000.0, 56000.0, 72900.0, 132533.33, 99823.53, 84800.0]
#### NULL COMPANIES AVG SALARY = 133447.01 ####
#### UNCATEGORIZED COMPANIES AVG SALARY  = 109095.75 ####


In [54]:
### Determine the maximum average salary among all companies
all_companies_slr_avg = np.concatenate((np.array(company_slr_avg), np.array(company_null_salary_avg),
        np.array(company_uncategorized_salary_avg)), axis=None)
all_companies_slr_max_avg = all_companies_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_companies_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 296363.0 ####


In [55]:
### Make a dictionary to store all salary coefficients of all companies; assign those coefficients as a new feature
company_salary_avg_dict = {}
for i in range(len(company_slr_avg)):
    company_salary_avg_dict["CMP"+str(i).zfill(2)] = round(company_slr_avg[i] / all_companies_slr_max_avg, 3)
# company_salary_avg_dict['NO_COMPANY'] = round(company_null_salary_avg / all_companies_slr_max_avg ,3)
company_salary_avg_dict['NO_COMPANY'] = 0.200
company_salary_avg_dict['UNCT'] = round(company_uncategorized_salary_avg / all_companies_slr_max_avg, 3)
# print(company_salary_avg_dict)

### Add these company codes and salary coefficients to the dataframe
for code, coef in company_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['company'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

company_slr_coef = [];
for code in salaries['company_code']:
    company_slr_coef.append(company_salary_avg_dict[code])
salaries['company_slr_coef'] = company_slr_coef

### 12. MAIN SKILLS

In [56]:
### Create main skills major code list to represent all the main skills as in short codes
main_skill_code = []
### Count each identified main skill for all rows, and collect their corresponding salaries
### Keep in mind that there is no case for main skills being null
main_skill_cnt = [0 for _ in range(80)]
main_skill_slr = [[] for _ in range(80)]
# print(main_skill_cnt)
# print(main_skill_slr)
# Uncategorized row numbers of main skills
main_skill_uncategorized = 0;   main_skill_uncategorized_salary = [];

In [57]:
all_main_skills = salaries['main_skills'].values
all_salaries = salaries['amount_usd'].values
for i in range(len(all_main_skills)):
    main_skills = all_main_skills[i];   salary = all_salaries[i];
    ms_code, indices = get_main_skills_code(main_skills)
    ms_code = ms_code.strip().replace(' ',',')
    main_skill_code.append(ms_code)
    if ms_code.startswith('MS'):
        for j in range(len(indices)):
            main_skill_cnt[indices[j]] += 1;   main_skill_slr[indices[j]].append(salary);
    elif ms_code == 'UNCT':
        main_skill_uncategorized += 1;   main_skill_uncategorized_salary.append(salary);
salaries['main_skill_code'] = main_skill_code

In [58]:
print("#### COUNTS OF ALL MAIN SKILLS ####")
print(main_skill_cnt, '-->', sum(main_skill_cnt))
print("#### {} UNCATEGORIZED MAIN SKILLS ####".format(main_skill_uncategorized))

#### COUNTS OF ALL MAIN SKILLS ####
[8845, 6512, 5725, 5808, 5574, 5343, 5331, 5198, 4867, 4825, 4116, 3832, 3829, 3588, 3461, 3498, 3443, 3268, 3154, 3105, 3087, 3137, 2969, 3069, 2727, 2709, 2651, 2431, 2382, 2339, 2366, 2332, 2280, 2277, 2273, 2259, 2150, 2160, 2149, 2147, 2119, 2107, 2030, 2024, 2038, 1873, 1822, 1825, 1914, 1848, 1673, 1580, 1605, 1559, 1429, 1372, 1387, 1388, 1328, 1337, 1275, 1223, 1233, 1232, 1180, 1165, 1151, 1161, 1148, 1140, 1154, 1055, 1007, 980, 1022, 952, 972, 943, 955, 963] --> 199385
#### 292 UNCATEGORIZED MAIN SKILLS ####


In [59]:
### Get the salary averages of all main skills
main_skill_slr_avg = [];
print("#### ALL MAIN SKILLS AVG SALARY ####")
for i in range(len(main_skill_slr)):
   main_skill_slr_avg.append(round(np.array(main_skill_slr[i]).mean(),2))
print(main_skill_slr_avg)

main_skill_uncategorized_salary_avg = round(np.array(main_skill_uncategorized_salary).mean(),2)
print("#### UNCATEGORIZED MAIN SKILLS AVG SALARY  = {} ####".format(main_skill_uncategorized_salary_avg))

#### ALL MAIN SKILLS AVG SALARY ####
[151429.78, 183265.1, 178546.42, 148697.37, 173176.67, 178077.08, 178102.77, 183430.67, 145676.44, 143509.66, 176992.64, 193922.8, 158887.63, 136028.54, 180346.35, 118010.89, 122391.15, 178906.53, 166029.67, 163649.94, 195755.35, 142547.61, 199899.99, 135922.58, 189420.06, 178849.69, 128538.26, 145669.7, 125686.42, 188569.84, 125497.87, 147220.03, 202185.61, 146433.08, 146507.32, 146880.09, 178804.27, 143322.86, 165779.41, 137752.56, 174966.42, 121651.11, 173720.78, 173594.74, 172581.97, 168812.82, 195838.18, 183228.55, 120932.42, 138485.39, 171945.51, 207282.44, 112341.75, 119110.52, 170072.8, 198518.42, 179820.66, 135097.56, 160848.95, 145484.43, 151003.17, 184431.21, 158994.89, 175360.3, 139761.13, 179640.1, 168955.43, 139975.14, 185272.04, 167203.26, 119443.23, 174154.74, 183807.26, 203742.45, 145614.09, 175609.49, 156453.85, 177489.39, 134529.42, 164633.71]
#### UNCATEGORIZED MAIN SKILLS AVG SALARY  = 153045.84 ####


In [60]:
### Determine the maximum average salary among all main skills
all_main_skills_slr_avg = np.concatenate((np.array(main_skill_slr_avg), np.array(main_skill_uncategorized_salary_avg)), axis=None)
all_main_skills_slr_max_avg = all_main_skills_slr_avg.max()
print("#### MAXIMUM AVERAGE SALARY = {} ####".format(all_main_skills_slr_max_avg))

#### MAXIMUM AVERAGE SALARY = 207282.44 ####


In [61]:
### Make a dictionary to store all salary coefficients of all main skills
### Assign those coefficients as two new features (average & maximum)
main_skill_salary_avg_dict = {}
for i in range(len(main_skill_slr_avg)):
    main_skill_salary_avg_dict["MS"+str(i).zfill(2)] = round(main_skill_slr_avg[i] / all_main_skills_slr_max_avg, 3)
main_skill_salary_avg_dict['UNCT'] = round(main_skill_uncategorized_salary_avg / all_main_skills_slr_max_avg, 3)
# print(main_skill_salary_avg_dict)

### Add these main skill codes and salary coefficients to the dataframe
for code, coef in main_skill_salary_avg_dict.items():
    new_line_df = pd.DataFrame({'feature': ['main_skill'], 'code': [code], 'coef': [coef]})
    features_codes_coefs_df = pd.concat([features_codes_coefs_df, new_line_df], ignore_index=True)

main_skill_slr_coef_avg = [];   main_skill_slr_coef_max = [];
for code in salaries['main_skill_code']:
    ms_codes = code.split(',');   coefs = []
    for ms in ms_codes:
        coefs.append(main_skill_salary_avg_dict[ms])
    main_skill_slr_coef_avg.append(round(np.array(coefs).mean(), 3))
    main_skill_slr_coef_max.append(round(np.array(coefs).max(), 3))
salaries['main_skill_slr_coef_avg'] = main_skill_slr_coef_avg
salaries['main_skill_slr_coef_max'] = main_skill_slr_coef_max

### THE FINAL DATASET

In [62]:
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd,country_coef,title_code,title_slr_coef,title_seniority,title_seniority_slr_coef,position_code,position_slr_coef,position_seniority,position_seniority_slr_coef,employment_type_coef,emp_type,employment_type_slr_coef,highest_edu_deg_name,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_code,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_code,company_slr_coef,main_skill_code,main_skill_slr_coef_avg,main_skill_slr_coef_max
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000,1.00,WC68,0.488,NO_SENIORITY,0.488,WC66,0.492,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,MASTERS,0.75,0.770,EDM17,0.796,0.796,UNCT,0.368,"MS01,MS02,MS03,MS04,MS05,MS06,MS07,MS11,MS17,M...",0.872,0.944
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000,1.00,UNCT,0.319,ASSOCIATE,0.319,UNCT,0.329,ASSOCIATE,0.340,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,NO_DEGREE,0.200,0.200,NO_COMPANY,0.200,"MS00,MS13,MS15,MS16,MS28,MS30,MS31,MS33,MS34,M...",0.646,0.731
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191,0.61,WC76,0.399,NO_SENIORITY,0.488,WC67,0.400,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,"EDM14,EDM30,EDM38,EDM53",0.779,0.823,NO_COMPANY,0.200,"MS00,MS16,MS40,MS42,MS43,MS45,MS62",0.774,0.844
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000,1.00,WC36,0.955,NO_SENIORITY,0.488,WC32,0.959,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM14,0.823,0.823,CMP01,0.845,"MS20,MS42,MS43",0.873,0.944
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000,1.00,WC07,0.503,NO_SENIORITY,0.488,WC06,0.516,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM19,0.655,0.655,CMP00,0.593,"MS03,MS08,MS12,MS14,MS19",0.769,0.870


In [63]:
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd,country_coef,title_slr_coef,title_seniority_slr_coef,position_slr_coef,position_seniority_slr_coef,employment_type_coef,employment_type_slr_coef,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_slr_coef,main_skill_slr_coef_avg,main_skill_slr_coef_max
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981,0.966054,0.452360,0.511637,0.456774,0.550829,0.651904,0.701566,0.612090,0.705434,0.695443,0.714984,0.517684,0.784424,0.884132
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433,0.109942,0.145708,0.074651,0.148300,0.087546,0.426533,0.382863,0.178708,0.128333,0.147528,0.158501,0.206434,0.083624,0.089182
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000,0.610000,0.102000,0.200000,0.115000,0.248000,0.100000,0.200000,0.100000,0.200000,0.200000,0.200000,0.181000,0.569000,0.569000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000,1.000000,0.350000,0.488000,0.338000,0.518000,0.100000,0.200000,0.500000,0.657000,0.591000,0.603000,0.368000,0.712000,0.833000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000,1.000000,0.484000,0.488000,0.492000,0.518000,0.990000,1.000000,0.500000,0.657000,0.732000,0.749000,0.541000,0.801000,0.894000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000,1.000000,0.539000,0.488000,0.565000,0.518000,0.990000,1.000000,0.750000,0.770000,0.796000,0.806000,0.676000,0.856000,0.964000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.990000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [64]:
salaries.isnull().sum()

id                                  0
found_country                       0
title                              12
position                            0
employment_type                  7157
company                          2618
company_score                    3815
edu_degrees                       554
edu_degrees_major                 931
working_year                        0
education_score                  4598
ms_counts                           0
skill_counts                        0
main_skills                         0
skills                              0
amount_usd                          0
country_coef                        0
title_code                          0
title_slr_coef                      0
title_seniority                     0
title_seniority_slr_coef            0
position_code                       0
position_slr_coef                   0
position_seniority                  0
position_seniority_slr_coef         0
employment_type_coef                0
emp_type    

In [65]:
### Inspect the dataframe that holds all features codes and their corresponding salary coefficients
features_codes_coefs_df

,feature,code,coef
0,title,WC00,0.607
1,title,WC01,0.477
2,title,WC02,0.484
3,title,WC03,0.697
4,title,WC04,0.797
...,...,...,...
505,main_skill,MS76,0.755
506,main_skill,MS77,0.856
507,main_skill,MS78,0.649
508,main_skill,MS79,0.794


In [66]:
### Save this dataset with user-defined extended features
# salaries.to_csv('Salaries_v4_202412161100_enhanced.csv')
# print("Saved dataset to disk...")
### Also save the feature codes and their coefficients
# features_codes_coefs_df.to_csv('features_codes_coefs.csv')
# print("Saved features, codes & coefficients to disk...")